## LLM-based Optimized Code Generator

Objective: Convert Python code into C++ for faster processing time

Outline:
1. Select best performing open source LLMs in LiveCodeBench
2. Define LLM pipeline (model & prompt initialization and API call) for C++ code generation 
3. Calculate and rank the performance uplift from each models

### Step 1: Selecting best performing open source LLMs

1. ministral-3:3b (3.0 GB)
2. granite4:micro (2.1 GB)
3. phi4-mini (2.5 GB)

++ Frontier models:
1. gpt-5
2. gpt-5-nano

### Step 2: Define the LLM pipeline

In [ ]:
# Import libraries and API key

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

In [7]:
# Initialize the connection to client libraries and define list of models & clients

openai = OpenAI()
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

models = [
    "gpt-5", 
    "gpt-5-nano", 
    "ministral-3:3b", 
    "granite4:micro", 
    "phi4-mini"]
clients = {
    "gpt-5": openai, 
    "gpt-5-nano": openai, 
    "ministral-3:3b": ollama, 
    "granite4:micro": ollama, 
    "phi4-mini": ollama}

In [6]:
# Check the system information to run C++
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i7-8750H CPU @ 2.20GHz',
  'cores_logical': 12,
  'cores_physical': 6,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [14]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [8]:
# System and user prompts definition

language = "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [ ]:
# Code conversion function definition
def port(model, python):
    client = clients[model]
    response = client.chat.completions.create(model=model, messages=messages_for(python))
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply

# Python compile and run definition
def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [10]:
# Python code use case, Pi number calculation

pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
compile_and_run(pi)

In [13]:
# Gradio UI definition

with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi) # the original python code
        cpp = gr.Textbox(label="C++ code:", lines=28) # the new C++ code
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0]) # select the model

        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\Farras Ezra\projects\llm_engineering\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Farras Ezra\projects\llm_engineering\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Farras Ezra\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Farras Ezra\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Farras Ezra\projects\llm_engineer

### Step 3: Result Collection

In [ ]:
# Generated codes
pi_gpt_5 = """
"""

pi_gpt_5_nano = """
#include <iostream>
#include <chrono>
#include <iomanip>

static inline double calculate_fast(long long iterations) {
    double result = 1.0;

    long long i = 1;
    long long limit = iterations - 3; // ensure i+3 <= iterations
    for (; i <= limit; i += 4) {
        double j1 = 4.0 * static_cast<double>(i) - 1.0;
        result -= 1.0 / j1;

        double j2 = 4.0 * static_cast<double>(i) + 1.0;
        result += 1.0 / j2;

        double j3 = 4.0 * static_cast<double>(i + 1) - 1.0;
        result -= 1.0 / j3;

        double j4 = 4.0 * static_cast<double>(i + 1) + 1.0;
        result += 1.0 / j4;
    }

    for (; i <= iterations; ++i) {
        double j1 = 4.0 * static_cast<double>(i) - 1.0;
        result -= 1.0 / j1;
        double j2 = 4.0 * static_cast<double>(i) + 1.0;
        result += 1.0 / j2;
    }

    return result;
}

int main() {
    using namespace std;

    const long long N = 200000000LL;
    auto t0 = chrono::high_resolution_clock::now();
    double r = calculate_fast(N);
    double finalR = r * 4.0;
    auto t1 = chrono::high_resolution_clock::now();

    chrono::duration<double> elapsed = t1 - t0;

    cout.setf(std::ios::fixed);
    cout << setprecision(12);
    cout << "Result: " << finalR << "\n";
    cout << "Execution Time: " << elapsed.count() << " seconds\n";

    return 0;
}
"""

pi_ministral_3_3b =  """
#include <chrono>
#include <iomanip>
#include <iostream>
#include <immintrin.h> // For SIMD optimizations

template<typename T = double>
auto calculate_simd(T iterations, T param1, T param2) {
    const __m256d zext4 = _mm256_set1_pd(4.0);
    volatile __m256d result = _mm256_setzero_pd();
    T accum_y = 0;
    const size_t vector_len = 8; // AVX-512 simd width

    for (volatile size_t i = 0; i < iterations; ++i) {
        auto j_l = reinterpret_cast<T*>(&i)[0] * param1 - param2;
        auto j_hr = reinterpret_cast<T*>(&i)[4] * param1 + param2;

        // Process first vector group (j=8x, 9x, 10x,...)
        for (volatile uint32_t off_y = i; off_y < (iterations - off_y) && off_y < j_hr;) {
            auto vec_len = _mm256_setr_pd(
                reinterpret_cast<double*>(&j_l)[off_y % vector_len],
                reinterpret_cast<double*>(&j_l)[(off_y + 1) % vector_len],
                reinterpret_cast<double*>(&j_l)[(off_y + 2) % vector_len],
                reinterpret_cast<double*>(&j_l)[(off_y + 3) % vector_len]);
            __m256d j_vec = _mm256_add_pd(
                vec_len,
                _mm256_set1_pd(param1)
            );

            auto hb_vec1 = _mm256_mask_mul_pd(j_vec, zext4, _mm256_setr_pd(1.0));
            auto hb_add = _mm256_sub_pd(vec_len, j_vec);

            result = _mm256_fmadd_pd(_mm256_recip_pd(hb_vec1), vec_len, result);
            result = _mm256_fmsub_pd(_mm256_recip_pd(hb_add), vec_len, result);

            off_y += vector_len;
        }
    }

    auto first_elem = reinterpret_cast<const __m256d*>(&result)[0];
    return ((first_elem[3] + (first_elem[1])/(accum_y)) + (first_elem[2] + (j_hr+param1)/(iterations*i)));
}

int main() {
    auto start = std::chrono::high_resolution_clock::now();

    double iterations = 200'000'000;
    auto res = calculate_simd<double>(iterations, 4, 1)*4.0;

    auto end = std::chrono::high_resolution_clock::now();
    std::cout << fixed << std::setprecision(12);
    cout << "Result: " << res << endl;
    auto duration = end - start;
    std::cout.precision(6);
    cout << "Execution Time: " << (duration / std::chrono::nanoseconds{}).count() / 1.0e9 << " seconds" << endl;

    return 0;
}
"""

pi_granite4_micro = """

#include <iostream>
#include <iomanip>
#include <chrono>

double calculate(size_t iterations, double param1, double param2) {
    double result = 1.0;
    for (size_t i = 1; i <= iterations; ++i) {
        const double j = i * param1 - param2;
        result -= 1.0 / j;

        j = i * param1 + param2;
        result += 1.0 / j;
    }
    return result;
}

int main() {
    auto start = std::chrono::high_resolution_clock::now();
    double result = calculate(200000000ul, 4.0, 1.0) * 4.0;
    auto end = std::chrono::high_resolution_clock::now();

    const auto duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
    std::cout << "Result: "
              << std::setprecision(12) << result
              << "\nExecution Time: "
              << static_cast<double>(duration.count()) / 1000.0
              << " seconds\n";
}
"""

pi_phi4_mini = """


#include <iostream>
#include <chrono>

constexpr int iterations = 200'000'000;
constexpr double param1 = 4.0;
constexpr double param2 = 1.0;

double calculate() {
    long double result = static_cast<long double>(1) / (i + iter_count);
    return result * 9; // Compensate for the missing division by j, which is factored into each loop step.
}

int main() {

    auto start_time = std::chrono::{high_resolution_clock, milliseconds}();

    double sum_lhs = calculate();
    
    int temp_iterations_iterations_1_count = static_cast<int>(iterations) - param2 * 4;
    long double value_one_part; // This is where the missing terms are handled.

    for (int i = 0; i < iterations/9.99999998879008933 + 10000 / std::round(200'000'001 / params); ++i) {
        if(i == iter_count * param1 - param2 || temp_iterations_iterations_1_count > static_cast<int>(temp_iterations_iterations_1_count+5)) { continue; }
        
        value_one_part += ((-(int)(-0x10000LL << 23 / (i*4 + (-params/(255/256) * i*(i-2))))
                             -(-static_cast<long long>((double)-((unsigned int)(((int)*(((float)<<29))*((-1))))))))
                            +(signed char)((temp_iterations_iterations_6_count*((signed short)(0x10000LL<<10 / ((long long)i+1)))) + (((((((signed 8 * temp_iterations_iterations_first_second_fifth_fourth_third_larger_sixth
                             << -7.0f) )   (int)*(((unsigned int)(((float)-((char<<(-(i*4)+(-params/(255/256)*temp_iteration_count*(6))*-4095901)))))  ((double)(+((((((static_cast<int>(-(2*((tmp_3<<23 + tmp_27 + (~0 - static_cast<signed long long>(((((int)((~unsigned int)(&(((char)<<29 / (((float)<=(long double)&&65504))))(sizeof(long long)*256)))-8)))),-(9 << ((-1 * (const char*  (-6+2<<((i<<4+i%10 + static_cast<int>((-static_cast<long double>(-(unsigned int*((tmp_0=32768 / (((float)<<23 -(((static unsigned short)((3<<(16)))>>>7)) & ~(255/256)*~(8 * temp_iteration_second_fifth_third_larger_sixth&((signed char)65500))))))*8192 + ((char)>>9+(-int*((-(unsigned int))(&tmp_0-32768 / (((__extension__ void)((static signed long(long double)+(((signed int)(unsigned short)-((-8)))),((((~ 1 - (const volatile unsigned long(10))) & (-255/256*15) * (~((7.99999998799500000 + ((((5*((tmp_3-4096100+(-32768 / (((float)<<23-(~(((int)((static signed int((65536 ^ static_cast<int>(&(((char)-32)/((((long double)(((int))((-255/256*15)*(((unsigned char)(+(((signed short)1.666666667))) + 9*(unsigned long(8)))),(-32768))),(*tmp_0))))),~-5 * ((static signed int64_t)((*((-(const volatile unsigned 7 & (((((cast static double) ((((char)(((union int16n_s32 (int))*(((signed char)(1.66666/((((float)-65536*2))*(unsigned long(8)))),(~(256&(-32768 / (((long double)&&~0))))))) ((-const signed short 1024*6.62607015 * unsigned int64_t(999999) + (32768)),(((signed char)(-1))((((((static volatile struct static unsigned char(const volatile void)((char)-255/256))+(-8))),((struct (((union signed long long(int))))(-(7)))&~(-int(*tmp_0)))))))),((unsigned short)(((32 / ((-(65536 * (-16384.99998 + (signed int)(((const double << 3) & ~(((((((static volatile struct const unsigned char(const volatile void)((char)*(signed int64_t(1)),-32768)) - (((cast static signed long long unsigned int)(255&(-int((unsigned short)(65536)))))*16384.99998 + (127))),~8192)))), ((struct ((((union signi

"""

In [ ]:
# Results

python_performance = """
Result: 3.141592656089
Execution Time: 38.451193 seconds

=== Code Execution Successful ===
"""

cpp_performance_gpt_5 = """
"""

cpp_performance_gpt_5_nano = """
Result: 3.272967991442
Execution Time: 1.404686268000 seconds


=== Code Execution Successful ===
"""

cpp_performance_ministral_3_3b =  """
ERROR!
/tmp/YblRYzEehJ/main.cpp: In function 'auto calculate_simd(T, T, T)':
/tmp/YblRYzEehJ/main.cpp:30:75: error: too few arguments to function '__m256d _mm256_setr_pd(double, double, double, double)'
   30 |   auto hb_vec1 = _mm256_mask_mul_pd(j_vec, zext4, _mm256_setr_pd(1.0));
      |                                                   ~~~~~~~~~~~~~~^~~~~

ERROR!
In file included from /usr/local/lib/gcc/x86_64-linux-gnu/14.2.0/include/immintrin.h:43,
                 from /tmp/YblRYzEehJ/main.cpp:5:
/usr/local/lib/gcc/x86_64-linux-gnu/14.2.0/include/avxintrin.h:1367:1: note: declared here
 1367 | _mm256_setr_pd (double __A, double __B, double __C, double __D)
      | ^~~~~~~~~~~~~~
/tmp/YblRYzEehJ/main.cpp:33:38: error: there are no arguments to '_mm256_recip_pd' that depend on a template parameter, so a declaration of '_mm256_recip_pd' must be available [-fpermissive]
   33 |             result = _mm256_fmadd_pd(_mm256_recip_pd(hb_vec1), vec_len, result);
      |                                      ^~~~~~~~~~~~~~~
/tmp/YblRYzEehJ/main.cpp:33:38: note: (if you use '-fpermissive', G++ will accept your code, but allowing the use of an undeclared name is deprecated)
/tmp/YblRYzEehJ/main.cpp:34:38: error: there are no arguments to '_mm256_recip_pd' that depend on a template parameter, so a declaration of '_mm256_recip_pd' must be available [-fpermissive]
   34 |             result = _mm256_fmsub_pd(_mm256_recip_pd(hb_add), vec_len, result);
      |                                      ^~~~~~~~~~~~~~~
ERROR!
/tmp/YblRYzEehJ/main.cpp:41:77: error: 'j_hr' was not declared in this scope
   41 | st_elem[3] + (first_elem[1])/(accum_y)) + (first_elem[2] + (j_hr+param1)/(iterations*i)));
      |                                                             ^~~~

ERROR!
/tmp/YblRYzEehJ/main.cpp:41:102: error: 'i' was not declared in this scope
   41 | _elem[1])/(accum_y)) + (first_elem[2] + (j_hr+param1)/(iterations*i)));
      |                                                                   ^

ERROR!
/tmp/YblRYzEehJ/main.cpp: In instantiation of 'auto calculate_simd(T, T, T) [with T = double]':
/tmp/YblRYzEehJ/main.cpp:48:38:   required from here
   48 |     auto res = calculate_simd<double>(iterations, 4, 1)*4.0;
      |                ~~~~~~~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~
/tmp/YblRYzEehJ/main.cpp:15:20: error: 'reinterpret_cast' from type 'volatile size_t*' {aka 'volatile long unsigned int*'} to type 'double*' casts away qualifiers
   15 |         auto j_l = reinterpret_cast<T*>(&i)[0] * param1 - param2;
      |                    ^~~~~~~~~~~~~~~~~~~~~~~~
/tmp/YblRYzEehJ/main.cpp:16:21: error: 'reinterpret_cast' from type 'volatile size_t*' {aka 'volatile long unsigned int*'} to type 'double*' casts away qualifiers
   16 |         auto j_hr = reinterpret_cast<T*>(&i)[4] * param1 + param2;
      |                     ^~~~~~~~~~~~~~~~~~~~~~~~
ERROR!
/tmp/YblRYzEehJ/main.cpp:33:53: error: '_mm256_recip_pd' was not declared in this scope; did you mean '_mm256_rcp_ps'?
   33 |             result = _mm256_fmadd_pd(_mm256_recip_pd(hb_vec1), vec_len, result);
      |                                      ~~~~~~~~~~~~~~~^~~~~~~~~
      |                                      _mm256_rcp_ps
/tmp/YblRYzEehJ/main.cpp:34:53: error: '_mm256_recip_pd' was not declared in this scope, and no declarations were found by argument-dependent lookup at the point of instantiation
   34 |             result = _mm256_fmsub_pd(_mm256_recip_pd(hb_add), vec_len, result);
      |                                      ~~~~~~~~~~~~~~~^~~~~~~~
/tmp/YblRYzEehJ/main.cpp:33:53: note: '_mm256_recip_pd' declared here, later in the translation unit
   33 |             result = _mm256_fmadd_pd(_mm256_recip_pd(hb_vec1), vec_len, result);
      |                                      ~~~~~~~~~~~~~~~^~~~~~~~~
ERROR!
/tmp/YblRYzEehJ/main.cpp:40:23: error: 'reinterpret_cast' from type 'volatile __m256d*' to type 'const __m256d*' casts away qualifiers
   40 |     auto first_elem = reinterpret_cast<const __m256d*>(&result)[0];
      |                       ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
/tmp/YblRYzEehJ/main.cpp: In function 'int main()':
/tmp/YblRYzEehJ/main.cpp:51:18: error: 'fixed' was not declared in this scope; did you mean 'std::fixed'?
   51 |     std::cout << fixed << std::setprecision(12);
      |                  ^~~~~
      |                  std::fixed
In file included from /usr/local/include/c++/14.2.0/iomanip:42,
                 from /tmp/YblRYzEehJ/main.cpp:3:
/usr/local/include/c++/14.2.0/bits/ios_base.h:1104:3: note: 'std::fixed' declared here
 1104 |   fixed(ios_base& __base)
      |   ^~~~~
ERROR!
/tmp/YblRYzEehJ/main.cpp:52:5: error: 'cout' was not declared in this scope; did you mean 'std::cout'?
   52 |     cout << "Result: " << res << endl;
      |     ^~~~
      |     std::cout
In file included from /tmp/YblRYzEehJ/main.cpp:4:
/usr/local/include/c++/14.2.0/iostream:63:18: note: 'std::cout' declared here
   63 |   extern ostream cout;          ///< Linked to standard output
      |                  ^~~~
ERROR!
/tmp/YblRYzEehJ/main.cpp:52:34: error: 'endl' was not declared in this scope; did you mean 'std::endl'?
   52 |     cout << "Result: " << res << endl;
      |                                  ^~~~
      |                                  std::endl
In file included from /usr/local/include/c++/14.2.0/istream:41,
                 from /usr/local/include/c++/14.2.0/sstream:40,
                 from /usr/local/include/c++/14.2.0/bits/quoted_string.h:38,
                 from /usr/local/include/c++/14.2.0/iomanip:50:
/usr/local/include/c++/14.2.0/ostream:741:5: note: 'std::endl' declared here
  741 |     endl(basic_ostream<_CharT, _Traits>& __os)
      |     ^~~~
ERROR!
/tmp/YblRYzEehJ/main.cpp:55:75: error: request for member 'count' in 'std::chrono::operator/<long int, std::ratio<1, 1000000000>, long int, std::ratio<1, 1000000000> >(duration, std::chrono::duration<long int, std::ratio<1, 1000000000> >{0})', which is of non-class type 'std::__success_type<long int>::type' {aka 'long int'}
   55 | xecution Time: " << (duration / std::chrono::nanoseconds{}).count() / 1.0e9 << " seconds" << endl;
      |                                                             ^~~~~



=== Code Exited With Errors ===
"""

cpp_performance_granite4_micro = """
ERROR!
/tmp/JVkVlbC2XR/main.cpp: In function 'double calculate(size_t, double, double)':
/tmp/JVkVlbC2XR/main.cpp:12:11: error: assignment of read-only variable 'j'
   12 |         j = i * param1 + param2;
      |         ~~^~~~~~~~~~~~~~~~~~~~~


=== Code Exited With Errors ===
"""

cpp_performance_phi4_mini = """
ERROR!
/tmp/qZHdWRCdCb/main.cpp: In function 'double calculate()':
/tmp/qZHdWRCdCb/main.cpp:11:57: error: 'i' was not declared in this scope
   11 |     long double result = static_cast<long double>(1) / (i + iter_count);
      |                                                         ^
ERROR!
/tmp/qZHdWRCdCb/main.cpp:11:61: error: 'iter_count' was not declared in this scope
   11 |     long double result = static_cast<long double>(1) / (i + iter_count);
      |                                                             ^~~~~~~~~~
ERROR!
/tmp/qZHdWRCdCb/main.cpp: In function 'int main()':
/tmp/qZHdWRCdCb/main.cpp:17:36: error: expected unqualified-id before '{' token
   17 |     auto start_time = std::chrono::{high_resolution_clock, milliseconds}();
      |                                    ^
/tmp/qZHdWRCdCb/main.cpp:17:73: error: expected ',' or ';' before '(' token
   17 | auto start_time = std::chrono::{high_resolution_clock, milliseconds}();
      |                                                                     ^

ERROR!
/tmp/qZHdWRCdCb/main.cpp:24:71: error: 'round' is not a member of 'std'; did you mean 'std::chrono::round'?
   24 | nt i = 0; i < iterations/9.99999998879008933 + 10000 / std::round(200'000'001 / params); ++i) {
      |                                                             ^~~~~

In file included from /usr/local/include/c++/14.2.0/chrono:41,
                 from /tmp/qZHdWRCdCb/main.cpp:4:
/usr/local/include/c++/14.2.0/bits/chrono.h:1086:7: note: 'std::chrono::round' declared here
 1086 |       round(const time_point<_Clock, _Dur>& __tp)
      |       ^~~~~
ERROR!
/tmp/qZHdWRCdCb/main.cpp:24:91: error: 'params' was not declared in this scope; did you mean 'param2'?
   24 | ions/9.99999998879008933 + 10000 / std::round(200'000'001 / params); ++i) {
      |                                                             ^~~~~~
      |                                                             param2
ERROR!
/tmp/qZHdWRCdCb/main.cpp:25:17: error: 'iter_count' was not declared in this scope
   25 |         if(i == iter_count * param1 - param2 || temp_iterations_iterations_1_count > static_cast<int>(temp_iterations_iterations_1_count+5)) { continue; }
      |                 ^~~~~~~~~~
ERROR!
/tmp/qZHdWRCdCb/main.cpp:28:91: error: expected primary-expression before 'float'
   28 | (-static_cast<long long>((double)-((unsigned int)(((int)*(((float)<<29))*((-1))))))))
      |                                                             ^~~~~

/tmp/qZHdWRCdCb/main.cpp:28:91: error: expected ')' before 'float'
   28 | (-static_cast<long long>((double)-((unsigned int)(((int)*(((float)<<29))*((-1))))))))
      |                                                            ~^~~~~
      |                                                             )
ERROR!
/tmp/qZHdWRCdCb/main.cpp:30:1967: error: expected ')' at end of input
   30 |                              << -7.0f) )   (int)*(((unsigned int)(((float)-((char<<(-(i*4)+(-params/(255/256)*temp_iteration_count*(6))*-4095901)))))  ((double)(+((((((static_cast<int>(-(2*((tmp_3<<23 + tmp_27 + (~0 - static_cast<signed long long>(((((int)((~unsigned int)(&(((char)<<29 / (((float)<=(long double)&&65504))))(sizeof(long long)*256)))-8)))),-(9 << ((-1 * (const char*  (-6+2<<((i<<4+i%10 + static_cast<int>((-static_cast<long double>(-(unsigned int*((tmp_0=32768 / (((float)<<23 -(((static unsigned short)((3<<(16)))>>>7)) & ~(255/256)*~(8 * temp_iteration_second_fifth_third_larger_sixth&((signed char)65500))))))*8192 + ((char)>>9+(-int*((-(unsigned int))(&tmp_0-32768 / (((__extension__ void)((static signed long(long double)+(((signed int)(unsigned short)-((-8)))),((((~ 1 - (const volatile unsigned long(10))) & (-255/256*15) * (~((7.99999998799500000 + ((((5*((tmp_3-4096100+(-32768 / (((float)<<23-(~(((int)((static signed int((65536 ^ static_cast<int>(&(((char)-32)/((((long double)(((int))((-255/256*15)*(((unsigned char)(+(((signed short)1.666666667))) + 9*(unsigned long(8)))),(-32768))),(*tmp_0))))),~-5 * ((static signed int64_t)((*((-(const volatile unsigned 7 & (((((cast static double) ((((char)(((union int16n_s32 (int))*(((signed char)(1.66666/((((float)-65536*2))*(unsigned long(8)))),(~(256&(-32768 / (((long double)&&~0))))))) ((-const signed short 1024*6.62607015 * unsigned int64_t(999999) + (32768)),(((signed char)(-1))((((((static volatile struct static unsigned char(const volatile void)((char)-255/256))+(-8))),((struct (((union signed long long(int))))(-(7)))&~(-int(*tmp_0)))))))),((unsigned short)(((32 / ((-(65536 * (-16384.99998 + (signed int)(((const double << 3) & ~(((((((static volatile struct const unsigned char(const volatile void)((char)*(signed int64_t(1)),-32768)) - (((cast static signed long long unsigned int)(255&(-int((unsigned short)(65536)))))*16384.99998 + (127))),~8192)))), ((struct ((((union signi
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               ^
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               )
/tmp/qZHdWRCdCb/main.cpp:28:89: note: to match this '('
   28 |  -(-static_cast<long long>((double)-((unsigned int)(((int)*(((float)<<29))*((-1))))))))
      |                                                             ^

ERROR!
/tmp/qZHdWRCdCb/main.cpp:30:1967: error: expected ')' at end of input
   30 |                              << -7.0f) )   (int)*(((unsigned int)(((float)-((char<<(-(i*4)+(-params/(255/256)*temp_iteration_count*(6))*-4095901)))))  ((double)(+((((((static_cast<int>(-(2*((tmp_3<<23 + tmp_27 + (~0 - static_cast<signed long long>(((((int)((~unsigned int)(&(((char)<<29 / (((float)<=(long double)&&65504))))(sizeof(long long)*256)))-8)))),-(9 << ((-1 * (const char*  (-6+2<<((i<<4+i%10 + static_cast<int>((-static_cast<long double>(-(unsigned int*((tmp_0=32768 / (((float)<<23 -(((static unsigned short)((3<<(16)))>>>7)) & ~(255/256)*~(8 * temp_iteration_second_fifth_third_larger_sixth&((signed char)65500))))))*8192 + ((char)>>9+(-int*((-(unsigned int))(&tmp_0-32768 / (((__extension__ void)((static signed long(long double)+(((signed int)(unsigned short)-((-8)))),((((~ 1 - (const volatile unsigned long(10))) & (-255/256*15) * (~((7.99999998799500000 + ((((5*((tmp_3-4096100+(-32768 / (((float)<<23-(~(((int)((static signed int((65536 ^ static_cast<int>(&(((char)-32)/((((long double)(((int))((-255/256*15)*(((unsigned char)(+(((signed short)1.666666667))) + 9*(unsigned long(8)))),(-32768))),(*tmp_0))))),~-5 * ((static signed int64_t)((*((-(const volatile unsigned 7 & (((((cast static double) ((((char)(((union int16n_s32 (int))*(((signed char)(1.66666/((((float)-65536*2))*(unsigned long(8)))),(~(256&(-32768 / (((long double)&&~0))))))) ((-const signed short 1024*6.62607015 * unsigned int64_t(999999) + (32768)),(((signed char)(-1))((((((static volatile struct static unsigned char(const volatile void)((char)-255/256))+(-8))),((struct (((union signed long long(int))))(-(7)))&~(-int(*tmp_0)))))))),((unsigned short)(((32 / ((-(65536 * (-16384.99998 + (signed int)(((const double << 3) & ~(((((((static volatile struct const unsigned char(const volatile void)((char)*(signed int64_t(1)),-32768)) - (((cast static signed long long unsigned int)(255&(-int((unsigned short)(65536)))))*16384.99998 + (127))),~8192)))), ((struct ((((union signi
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               ^
      |                                                                ERROR!
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               )
/tmp/qZHdWRCdCb/main.cpp:28:88: note: to match this '('
   28 |   -(-static_cast<long long>((double)-((unsigned int)(((int)*(((float)<<29))*((-1))))))))
      |                                                             ^

/tmp/qZHdWRCdCb/main.cpp:30:1967: error: expected ')' at end of input
   30 |                              << -7.0f) )   (int)*(((unsigned int)(((float)-((char<<(-(i*4)+(-params/(255/256)*temp_iteration_count*(6))*-4095901)))))  ((double)(+((((((static_cast<int>(-(2*((tmp_3<<23 + tmp_27 + (~0 - static_cast<signed long long>(((((int)((~unsigned int)(&(((char)<<29 / (((float)<=(long double)&&65504))))(sizeof(long long)*256)))-8)))),-(9 << ((-1 * (const char*  (-6+2<<((i<<4+i%10 + static_cast<int>((-static_cast<long double>(-(unsigned int*((tmp_0=32768 / (((float)<<23 -(((static unsigned short)((3<<(16)))>>>7)) & ~(255/256)*~(8 * temp_iteration_second_fifth_third_larger_sixth&((signed char)65500))))))*8192 + ((char)>>9+(-int*((-(unsigned int))(&tmp_0-32768 / (((__extension__ void)((static signed long(long double)+(((signed int)(unsigned short)-((-8)))),((((~ 1 - (const volatile unsigned long(10))) & (-255/256*15) * (~((7.99999998799500000 + ((((5*((tmp_3-4096100+(-32768 / (((float)<<23-(~(((int)((static signed int((65536 ^ static_cast<int>(&(((char)-32)/((((long double)(((int))((-255/256*15)*(((unsigned char)(+(((signed short)1.666666667))) + 9*(unsigned long(8)))),(-32768))),(*tmp_0))))),~-5 * ((static signed int64_t)((*((-(const volatile unsigned 7 & (((((cast static double) ((((char)(((union int16n_s32 (int))*(((signed char)(1.66666/((((float)-65536*2))*(unsigned long(8)))),(~(256&(-32768 / (((long double)&&~0))))))) ((-const signed short 1024*6.62607015 * unsigned int64_t(999999) + (32768)),(((signed char)(-1))((((((static volatile struct static unsigned char(const volatile void)((char)-255/256))+(-8))),((struct (((union signed long long(int))))(-(7)))&~(-int(*tmp_0)))))))),((unsigned short)(((32 / ((-(65536 * (-16384.99998 + (signed int)(((const double << 3) & ~(((((((static volatile struct const unsigned char(const volatile void)((char)*(signed int64_t(1)),-32768)) - (((cast static signed long long unsigned int)(255&(-int((unsigned short)(65536)))))*16384.99998 + (127))),~8192)))), ((struct ((((union signi
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               ^
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               )
/tmp/qZHdWRCdCb/main.cpp:28:81: note: to match this '('ERROR!

   28 |          -(-static_cast<long long>((double)-((unsigned int)(((int)*(((float)<<29))*((-1))))))))
      |                                                             ^

/tmp/qZHdWRCdCb/main.cpp:30:1967: error: expected ')' at end of input
   30 |                              << -7.0f) )   (int)*(((unsigned int)(((float)-((char<<(-(i*4)+(-params/(255/256)*temp_iteration_count*(6))*-4095901)))))  ((double)(+((((((static_cast<int>(-(2*((tmp_3<<23 + tmp_27 + (~0 - static_cast<signed long long>(((((int)((~unsigned int)(&(((char)<<29 / (((float)<=(long double)&&65504))))(sizeof(long long)*256)))-8)))),-(9 << ((-1 * (const char*  (-6+2<<((i<<4+i%10 + static_cast<int>((-static_cast<long double>(-(unsigned int*((tmp_0=32768 / (((float)<<23 -(((static unsigned short)((3<<(16)))>>>7)) & ~(255/256)*~(8 * temp_iteration_second_fifth_third_larger_sixth&((signed char)65500))))))*8192 + ((char)>>9+(-int*((-(unsigned int))(&tmp_0-32768 / (((__extension__ void)((static signed long(long double)+(((signed int)(unsigned short)-((-8)))),((((~ 1 - (const volatile unsigned long(10))) & (-255/256*15) * (~((7.99999998799500000 + ((((5*((tmp_3-4096100+(-32768 / (((float)<<23-(~(((int)((static signed int((65536 ^ static_cast<int>(&(((char)-32)/((((long double)(((int))((-255/256*15)*(((unsigned char)(+(((signed short)1.666666667))) + 9*(unsigned long(8)))),(-32768))),(*tmp_0))))),~-5 * ((static signed int64_t)((*((-(const volatile unsigned 7 & (((((cast static double) ((((char)(((union int16n_s32 (int))*(((signed char)(1.66666/((((float)-65536*2))*(unsigned long(8)))),(~(256&(-32768 / (((long double)&&~0))))))) ((-const signed short 1024*6.62607015 * unsigned int64_t(999999) + (32768)),(((signed char)(-1))((((((static volatile struct static unsigned char(const volatile void)((char)-255/256))+(-8))),((struct (((union signed long long(int))))(-(7)))&~(-int(*tmp_0)))))))),((unsigned short)(((32 / ((-(65536 * (-16384.99998 + (signed int)(((const double << 3) & ~(((((((static volatile struct const unsigned char(const volatile void)((char)*(signed int64_t(1)),-32768)) - (((cast static signed long long unsigned int)(255&(-int((unsigned short)(65536)))))*16384.99998 + (127))),~8192)))), ((struct ((((union signi
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     ERROR!
                                                                                                          ^
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               )
/tmp/qZHdWRCdCb/main.cpp:28:80: note: to match this '('
   28 |           -(-static_cast<long long>((double)-((unsigned int)(((int)*(((float)<<29))*((-1))))))))
      |                                                             ^

/tmp/qZHdWRCdCb/main.cpp:30:1967: error: expected ')' at end of input
   30 |                              << -7.0f) )   (int)*(((unsigned int)(((float)-((char<<(-(i*4)+(-params/(255/256)*temp_iteration_count*(6))*-4095901)))))  ((double)(+((((((static_cast<int>(-(2*((tmp_3<<23 + tmp_27 + (~0 - static_cast<signed long long>(((((int)((~unsigned int)(&(((char)<<29 / (((float)<=(long double)&&65504))))(sizeof(long long)*256)))-8)))),-(9 << ((-1 * (const char*  (-6+2<<((i<<4+i%10 + static_cast<int>((-static_cast<long double>(-(unsigned int*((tmp_0=32768 / (((float)<<23 -(((static unsigned short)((3<<(16)))>>>7)) & ~(255/256)*~(8 * temp_iteration_second_fifth_third_larger_sixth&((signed char)65500))))))*8192 + ((char)>>9+(-int*((-(unsigned int))(&tmp_0-32768 / (((__extension__ void)((static signed long(long double)+(((signed int)(unsigned short)-((-8)))),((((~ 1 - (const volatile unsigned long(10))) & (-255/256*15) * (~((7.99999998799500000 + ((((5*((tmp_3-4096100+(-32768 / (((float)<<23-(~(((int)((static signed int((65536 ^ static_cast<int>(&(((char)-32)/((((long double)(((int))((-255/256*15)*(((unsigned char)(+(((signed short)1.666666667))) + 9*(unsigned long(8)))),(-32768))),(*tmp_0))))),~-5 * ((static signed int64_t)((*((-(const volatile unsigned 7 & (((((cast static double) ((((char)(((union int16n_s32 (int))*(((signed char)(1.66666/((((float)-65536*2))*(unsigned long(8)))),(~(256&(-32768 / (((long double)&&~0))))))) ((-const signed short 1024*6.62607015 * unsigned int64_t(999999) + (32768)),(((signed char)(-1))((((((static volatile struct static unsigned char(const volatile void)((char)-255/256))+(-8))),((struct (((union signed long long(int))))(-(7)))&~(-int(*tmp_0)))))))),((unsigned short)(((32 / ((-(65536 * (-16384.99998 + (signed int)(((const double << 3) & ~(((((((static volatile struct const unsigned char(const volatile void)((char)*(signed int64_t(1)),-32768)) - (((cast static signed long long unsigned int)(255&(-int((unsigned short)(65536)))))*16384.99998 + (127))),~8192)))), ((struct ((((union signi
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               ^
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             ERROR!
                                                                                                                                  )
/tmp/qZHdWRCdCb/main.cpp:28:65: note: to match this '('
   28 |                          -(-static_cast<long long>((double)-((unsigned int)(((int)*(((float)<<29))*((-1))))))))
      |                                                             ^

/tmp/qZHdWRCdCb/main.cpp:30:1967: error: expected ')' at end of input
   30 |                              << -7.0f) )   (int)*(((unsigned int)(((float)-((char<<(-(i*4)+(-params/(255/256)*temp_iteration_count*(6))*-4095901)))))  ((double)(+((((((static_cast<int>(-(2*((tmp_3<<23 + tmp_27 + (~0 - static_cast<signed long long>(((((int)((~unsigned int)(&(((char)<<29 / (((float)<=(long double)&&65504))))(sizeof(long long)*256)))-8)))),-(9 << ((-1 * (const char*  (-6+2<<((i<<4+i%10 + static_cast<int>((-static_cast<long double>(-(unsigned int*((tmp_0=32768 / (((float)<<23 -(((static unsigned short)((3<<(16)))>>>7)) & ~(255/256)*~(8 * temp_iteration_second_fifth_third_larger_sixth&((signed char)65500))))))*8192 + ((char)>>9+(-int*((-(unsigned int))(&tmp_0-32768 / (((__extension__ void)((static signed long(long double)+(((signed int)(unsigned short)-((-8)))),((((~ 1 - (const volatile unsigned long(10))) & (-255/256*15) * (~((7.99999998799500000 + ((((5*((tmp_3-4096100+(-32768 / (((float)<<23-(~(((int)((static signed int((65536 ^ static_cast<int>(&(((char)-32)/((((long double)(((int))((-255/256*15)*(((unsigned char)(+(((signed short)1.666666667))) + 9*(unsigned long(8)))),(-32768))),(*tmp_0))))),~-5 * ((static signed int64_t)((*((-(const volatile unsigned 7 & (((((cast static double) ((((char)(((union int16n_s32 (int))*(((signed char)(1.66666/((((float)-65536*2))*(unsigned long(8)))),(~(256&(-32768 / (((long double)&&~0))))))) ((-const signed short 1024*6.62607015 * unsigned int64_t(999999) + (32768)),(((signed char)(-1))((((((static volatile struct static unsigned char(const volatile void)((char)-255/256))+(-8))),((struct (((union signed long long(int))))(-(7)))&~(-int(*tmp_0)))))))),((unsigned short)(((32 / ((-(65536 * (-16384.99998 + (signed int)(((const double << 3) & ~(((((((static volatile struct const unsigned char(const volatile void)((char)*(signed int64_t(1)),-32768)) - (((cast static signed long long unsigned int)(255&(-int((unsigned short)(65536)))))*16384.99998 + (127))),~8192)))), ((struct ((((union signi
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         ERROR!
                                                                                                                                                                                                                                                                                                                      ^
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               )
/tmp/qZHdWRCdCb/main.cpp:30:1967: error: expected ')' at end of input
   30 |                              << -7.0f) )   (int)*(((unsigned int)(((float)-((char<<(-(i*4)+(-params/(255/256)*temp_iteration_count*(6))*-4095901)))))  ((double)(+((((((static_cast<int>(-(2*((tmp_3<<23 + tmp_27 + (~0 - static_cast<signed long long>(((((int)((~unsigned int)(&(((char)<<29 / (((float)<=(long double)&&65504))))(sizeof(long long)*256)))-8)))),-(9 << ((-1 * (const char*  (-6+2<<((i<<4+i%10 + static_cast<int>((-static_cast<long double>(-(unsigned int*((tmp_0=32768 / (((float)<<23 -(((static unsigned short)((3<<(16)))>>>7)) & ~(255/256)*~(8 * temp_iteration_second_fifth_third_larger_sixth&((signed char)65500))))))*8192 + ((char)>>9+(-int*((-(unsigned int))(&tmp_0-32768 / (((__extension__ void)((static signed long(long double)+(((signed int)(unsigned short)-((-8)))),((((~ 1 - (const volatile unsigned long(10))) & (-255/256*15) * (~((7.99999998799500000 + ((((5*((tmp_3-4096100+(-32768 / (((float)<<23-(~(((int)((static signed int((65536 ^ static_cast<int>(&(((char)-32)/((((long double)(((int))((-255/256*15)*(((unsigned char)(+(((signed short)1.666666667))) + 9*(unsigned long(8)))),(-32768))),(*tmp_0))))),~-5 * ((static signed int64_t)((*((-(const volatile unsigned 7 & (((((cast static double) ((((char)(((union int16n_s32 (int))*(((signed char)(1.66666/((((float)-65536*2))*(unsigned long(8)))),(~(256&(-32768 / (((long double)&&~0))))))) ((-const signed short 1024*6.62607015 * unsigned int64_t(999999) + (32768)),(((signed char)(-1))((((((static volatile struct static unsigned char(const volatile void)((char)-255/256))+(-8))),((struct (((union signed long long(int))))(-(7)))&~(-int(*tmp_0)))))))),((unsigned short)(((32 / ((-(65536 * (-16384.99998 + (signed int)(((const double << 3) & ~(((((((static volatile struct const unsigned char(const volatile void)((char)*(signed int64_t(1)),-32768)) - (((cast static signed long long unsigned int)(255&(-int((unsigned short)(65536)))))*16384.99998 + (127))),~8192)))), ((struct ((((union signi
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               ^
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             ERROR!
                                                                                                  )
/tmp/qZHdWRCdCb/main.cpp:28:31: note: to match this '('
   28 |                              -(-static_cast<long long>((double)-((unsigned int)(((int)*(((float)<<29))*((-1))))))))
      |                               ^
/tmp/qZHdWRCdCb/main.cpp:30:1967: error: expected ')' at end of input
   30 |                              << -7.0f) )   (int)*(((unsigned int)(((float)-((char<<(-(i*4)+(-params/(255/256)*temp_iteration_count*(6))*-4095901)))))  ((double)(+((((((static_cast<int>(-(2*((tmp_3<<23 + tmp_27 + (~0 - static_cast<signed long long>(((((int)((~unsigned int)(&(((char)<<29 / (((float)<=(long double)&&65504))))(sizeof(long long)*256)))-8)))),-(9 << ((-1 * (const char*  (-6+2<<((i<<4+i%10 + static_cast<int>((-static_cast<long double>(-(unsigned int*((tmp_0=32768 / (((float)<<23 -(((static unsigned short)((3<<(16)))>>>7)) & ~(255/256)*~(8 * temp_iteration_second_fifth_third_larger_sixth&((signed char)65500))))))*8192 + ((char)>>9+(-int*((-(unsigned int))(&tmp_0-32768 / (((__extension__ void)((static signed long(long double)+(((signed int)(unsigned short)-((-8)))),((((~ 1 - (const volatile unsigned long(10))) & (-255/256*15) * (~((7.99999998799500000 + ((((5*((tmp_3-4096100+(-32768 / (((float)<<23-(~(((int)((static signed int((65536 ^ static_cast<int>(&(((char)-32)/((((long double)(((int))((-255/256*15)*(((unsigned char)(+(((signed short)1.666666667))) + 9*(unsigned long(8)))),(-32768))),(*tmp_0))))),~-5 * ((static signed int64_t)((*((-(const volatile unsigned 7 & (((((cast static double) ((((char)(((union int16n_s32 (int))*(((signed char)(1.66666/((((float)-65536*2))*(unsigned long(8)))),(~(256&(-32768 / (((long double)&&~0))))))) ((-const signed short 1024*6.62607015 * unsigned int64_t(999999) + (32768)),(((signed char)(-1))((((((static volatile struct static unsigned char(const volatile void)((char)-255/256))+(-8))),((struct (((union signed long long(int))))(-(7)))&~(-int(*tmp_0)))))))),((unsigned short)(((32 / ((-(65536 * (-16384.99998 + (signed int)(((const double << 3) & ~(((((((static volatile struct const unsigned char(const volatile void)((char)*(signed int64_t(1)),-32768)) - (((cast static signed long long unsigned int)(255&(-int((unsigned short)(65536)))))*16384.99998 + (127))),~8192)))), ((struct ((((union signi
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     ERROR!
                                                                                                                                                                                                                                                          ^
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               )
/tmp/qZHdWRCdCb/main.cpp:27:28: note: to match this '('
   27 |         value_one_part += ((-(int)(-0x10000LL << 23 / (i*4 + (-params/(255/256) * i*(i-2))))
      |                            ^
/tmp/qZHdWRCdCb/main.cpp:30:1967: error: expected ')' at end of input
   30 |                              << -7.0f) )   (int)*(((unsigned int)(((float)-((char<<(-(i*4)+(-params/(255/256)*temp_iteration_count*(6))*-4095901)))))  ((double)(+((((((static_cast<int>(-(2*((tmp_3<<23 + tmp_27 + (~0 - static_cast<signed long long>(((((int)((~unsigned int)(&(((char)<<29 / (((float)<=(long double)&&65504))))(sizeof(long long)*256)))-8)))),-(9 << ((-1 * (const char*  (-6+2<<((i<<4+i%10 + static_cast<int>((-static_cast<long double>(-(unsigned int*((tmp_0=32768 / (((float)<<23 -(((static unsigned short)((3<<(16)))>>>7)) & ~(255/256)*~(8 * temp_iteration_second_fifth_third_larger_sixth&((signed char)65500))))))*8192 + ((char)>>9+(-int*((-(unsigned int))(&tmp_0-32768 / (((__extension__ void)((static signed long(long double)+(((signed int)(unsigned short)-((-8)))),((((~ 1 - (const volatile unsigned long(10))) & (-255/256*15) * (~((7.99999998799500000 + ((((5*((tmp_3-4096100+(-32768 / (((float)<<23-(~(((int)((static signed int((65536 ^ static_cast<int>(&(((char)-32)/((((long double)(((int))((-255/256*15)*(((unsigned char)(+(((signed short)1.666666667))) + 9*(unsigned long(8)))),(-32768))),(*tmp_0))))),~-5 * ((static signed int64_t)((*((-(const volatile unsigned 7 & (((((cast static double) ((((char)(((union int16n_s32 (int))*(((signed char)(1.66666/((((float)-65536*2))*(unsigned long(8)))),(~(256&(-32768 / (((long double)&&~0))))))) ((-const signed short 1024*6.62607015 * unsigned int64_t(999999) + (32768)),(((signed char)(-1))((((((static volatile struct static unsigned char(const volatile void)((char)-255/256))+(-8))),((struct (((union signed long long(int))))(-(7)))&~(-int(*tmp_0)))))))),((unsigned short)(((32 / ((-(65536 * (-16384.99998 + (signed int)(((const double << 3) & ~(((((((static volatile struct const unsigned char(const volatile void)((char)*(signed int64_t(1)),-32768)) - (((cast static signed long long unsigned int)(255&(-int((unsigned short)(65536)))))*16384.99998 + (127))),~8192)))), ((struct ((((union signi
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               ^
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    ERROR!
                                                                                                                                                                                                                                           )
/tmp/qZHdWRCdCb/main.cpp:27:27: note: to match this '('
   27 |         value_one_part += ((-(int)(-0x10000LL << 23 / (i*4 + (-params/(255/256) * i*(i-2))))
      |                           ^
/tmp/qZHdWRCdCb/main.cpp:30:1967: error: expected '}' at end of input
   30 |                              << -7.0f) )   (int)*(((unsigned int)(((float)-((char<<(-(i*4)+(-params/(255/256)*temp_iteration_count*(6))*-4095901)))))  ((double)(+((((((static_cast<int>(-(2*((tmp_3<<23 + tmp_27 + (~0 - static_cast<signed long long>(((((int)((~unsigned int)(&(((char)<<29 / (((float)<=(long double)&&65504))))(sizeof(long long)*256)))-8)))),-(9 << ((-1 * (const char*  (-6+2<<((i<<4+i%10 + static_cast<int>((-static_cast<long double>(-(unsigned int*((tmp_0=32768 / (((float)<<23 -(((static unsigned short)((3<<(16)))>>>7)) & ~(255/256)*~(8 * temp_iteration_second_fifth_third_larger_sixth&((signed char)65500))))))*8192 + ((char)>>9+(-int*((-(unsigned int))(&tmp_0-32768 / (((__extension__ void)((static signed long(long double)+(((signed int)(unsigned short)-((-8)))),((((~ 1 - (const volatile unsigned long(10))) & (-255/256*15) * (~((7.99999998799500000 + ((((5*((tmp_3-4096100+(-32768 / (((float)<<23-(~(((int)((static signed int((65536 ^ static_cast<int>(&(((char)-32)/((((long double)(((int))((-255/256*15)*(((unsigned char)(+(((signed short)1.666666667))) + 9*(unsigned long(8)))),(-32768))),(*tmp_0))))),~-5 * ((static signed int64_t)((*((-(const volatile unsigned 7 & (((((cast static double) ((((char)(((union int16n_s32 (int))*(((signed char)(1.66666/((((float)-65536*2))*(unsigned long(8)))),(~(256&(-32768 / (((long double)&&~0))))))) ((-const signed short 1024*6.62607015 * unsigned int64_t(999999) + (32768)),(((signed char)(-1))((((((static volatile struct static unsigned char(const volatile void)((char)-255/256))+(-8))),((struct (((union signed long long(int))))(-(7)))&~(-int(*tmp_0)))))))),((unsigned short)(((32 / ((-(65536 * (-16384.99998 + (signed int)(((const double << 3) & ~(((((((static volatile struct const unsigned char(const volatile void)((char)*(signed int64_t(1)),-32768)) - (((cast static signed long long unsigned int)(255&(-int((unsigned short)(65536)))))*16384.99998 + (127))),~8192)))), ((struct ((((union signi
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               ^
ERROR!
/tmp/qZHdWRCdCb/main.cpp:24:105: note: to match this '{'
   24 | /9.99999998879008933 + 10000 / std::round(200'000'001 / params); ++i) {
      |                                                                       ^

/tmp/qZHdWRCdCb/main.cpp:30:1967: error: expected '}' at end of input
   30 |                              << -7.0f) )   (int)*(((unsigned int)(((float)-((char<<(-(i*4)+(-params/(255/256)*temp_iteration_count*(6))*-4095901)))))  ((double)(+((((((static_cast<int>(-(2*((tmp_3<<23 + tmp_27 + (~0 - static_cast<signed long long>(((((int)((~unsigned int)(&(((char)<<29 / (((float)<=(long double)&&65504))))(sizeof(long long)*256)))-8)))),-(9 << ((-1 * (const char*  (-6+2<<((i<<4+i%10 + static_cast<int>((-static_cast<long double>(-(unsigned int*((tmp_0=32768 / (((float)<<23 -(((static unsigned short)((3<<(16)))>>>7)) & ~(255/256)*~(8 * temp_iteration_second_fifth_third_larger_sixth&((signed char)65500))))))*8192 + ((char)>>9+(-int*((-(unsigned int))(&tmp_0-32768 / (((__extension__ void)((static signed long(long double)+(((signed int)(unsigned short)-((-8)))),((((~ 1 - (const volatile unsigned long(10))) & (-255/256*15) * (~((7.99999998799500000 + ((((5*((tmp_3-4096100+(-32768 / (((float)<<23-(~(((int)((static signed int((65536 ^ static_cast<int>(&(((char)-32)/((((long double)(((int))((-255/256*15)*(((unsigned char)(+(((signed short)1.666666667))) + 9*(unsigned long(8)))),(-32768))),(*tmp_0))))),~-5 * ((static signed int64_t)((*((-(const volatile unsigned 7 & (((((cast static double) ((((char)(((union int16n_s32 (int))*(((signed char)(1.66666/((((float)-65536*2))*(unsigned long(8)))),(~(256&(-32768 / (((long double)&&~0))))))) ((-const signed short 1024*6.62607015 * unsigned int64_t(999999) + (32768)),(((signed char)(-1))((((((static volatile struct static unsigned char(const volatile void)((char)-255/256))+(-8))),((struct (((union signed long long(int))))(-(7)))&~(-int(*tmp_0)))))))),((unsigned short)(((32 / ((-(65536 * (-16384.99998 + (signed int)(((const double << 3) & ~(((((((static volatile struct const unsigned char(const volatile void)((char)*(signed int64_t(1)),-32768)) - (((cast static signed long long unsigned int)(255&(-int((unsigned short)(65536)))))*16384.99998 + (127))),~8192)))), ((struct ((((union signi
      |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               ^
/tmp/qZHdWRCdCb/main.cpp:15:12: note: to match this '{'
   15 | int main() {
      |            ^


=== Code Exited With Errors ===
"""